<a href="https://colab.research.google.com/github/keiiigo/RecursosOpenSource/blob/capacitatedclustering/Capacitated_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clientescor.csv to clientescor.csv


In [3]:
import math
import numpy as np
import pandas as pd
import folium
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from google.colab import files
from IPython.display import display

In [4]:
#Lector de pd de nuestro csv
customer_data = pd.read_csv('clientescor.csv')

customer_data['Latitude'] = customer_data['Latitude'].astype(float)
customer_data['Longitude'] = customer_data['Longitude'].astype(float)

depot_lat = -25.3775625
depot_lon = -57.4761875
depot = [0, depot_lat, depot_lon, 0]

customers = [[i + 1, row['Customer Name'], row['Latitude'], row['Longitude'], row['Demand']]
             for i, row in customer_data.iterrows()]

X = customer_data[['Latitude', 'Longitude', 'Demand']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Capacidad de vehiculos
vehicle_capacity = 40

In [6]:
def capacitated_clustering():
    kmeans = KMeans(init='k-means++', n_init=10, max_iter=300)
    kmeans.fit(X_scaled)

    customer_data['Cluster'] = kmeans.labels_

    cluster_demands = customer_data.groupby('Cluster')['Demand'].sum()

    for cluster, demand in cluster_demands.items():
        if demand > vehicle_capacity:
            print(f"Cluster {cluster} exceeds capacity! Total demand: {demand}.")

            over_capacity_customers = customer_data[customer_data['Cluster'] == cluster]

            # Los clusters que esten por encima de la demanda son divididos en clusters mas pequenos
            sub_cluster_count = int(demand // vehicle_capacity) + 1

            sub_kmeans = KMeans(n_clusters=sub_cluster_count, init='k-means++', n_init=10, max_iter=300)
            sub_kmeans.fit(over_capacity_customers[['Latitude', 'Longitude', 'Demand']])

            customer_data.loc[customer_data['Cluster'] == cluster, 'Cluster'] = sub_kmeans.labels_ + max(kmeans.labels_) + 1
            print(f"Cluster {cluster} split into {sub_cluster_count} new clusters.")

    return customer_data['Cluster'].values

In [7]:
#Formula de Haversine, no cambiar!
def haversine(p1, p2):
    R = 6371.0

    lat1, lon1 = p1
    lat2, lon2 = p2

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lat = math.radians(lat2 - lat1)
    delta_lon = math.radians(lon2 - lon1)

    a = (math.sin(delta_lat / 2) ** 2 +
         math.cos(lat1_rad) * math.cos(lat2_rad) *
         (math.sin(delta_lon / 2) ** 2))
    c = 2 * math.asin(math.sqrt(a))
    return R * c

In [8]:
def find_routes():
    routes = []
    distances = []

    labels = capacitated_clustering()

    for i in set(labels):
        cluster_customers = [customers[j] for j in range(len(customers)) if labels[j] == i]

        if not cluster_customers:
            continue

        depot_location = (depot_lat, depot_lon)
        route = [("Depot", depot_location)]
        unvisited = cluster_customers[:]
        total_distance = 0

        while unvisited:
            last_visited = route[-1][1]
            nearest = min(unvisited, key=lambda c: haversine(last_visited, (c[2], c[3])))
            route.append((nearest[0], (nearest[2], nearest[3])))
            total_distance += haversine(last_visited, (nearest[2], nearest[3]))
            unvisited.remove(nearest)

        route.append(("Depot", depot_location))
        total_distance += haversine(route[-2][1], depot_location)

        routes.append(route)
        distances.append(total_distance)

    return routes, distances

In [9]:
def plot_routes():
    routes, distances = find_routes()

    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=12)

    COLORS = [ "green", "white", "beige", "lightred", "darkgreen", "lightblue", "pink",
    "red", "lightgreen", "cadetblue", "purple", "darkred", "orange",
    "blue", "gray", "darkblue", "darkpurple", "black", "lightgray"]

    for i, route in enumerate(routes):
        cluster_color = COLORS[i % len(COLORS)]

        for j in range(len(route) - 1):
            folium.Marker(
                location=route[j][1],
                popup=f'Stop {j}: {route[j][0]} (Cluster {i})',
                icon=folium.Icon(color=cluster_color)
            ).add_to(m)

        folium.PolyLine([stop[1] for stop in route], color=cluster_color, weight=2.5, opacity=0.8).add_to(m)

    folium.Marker(
        location=[depot_lat, depot_lon],
        popup='Depot',
        icon=folium.Icon(color='black', icon='info-sign')
    ).add_to(m)

    for i, (route, total_distance) in enumerate(zip(routes, distances)):
        print(f"Optimal Route for Cluster {i}:")
        for j, stop in enumerate(route):
            print(f"  Stop {j}: {stop[0]}")
        print(f"Total distance: {total_distance:.2f} km\n")

    return m

In [16]:
display(map_output)

In [17]:
map_output = plot_routes()

Optimal Route for Cluster 0:
  Stop 0: Depot
  Stop 1: 56
  Stop 2: 49
  Stop 3: 88
  Stop 4: 93
  Stop 5: 47
  Stop 6: 75
  Stop 7: 77
  Stop 8: 31
  Stop 9: 64
  Stop 10: 34
  Stop 11: 39
  Stop 12: 94
  Stop 13: 83
  Stop 14: 100
  Stop 15: 61
  Stop 16: Depot
Total distance: 35.14 km

Optimal Route for Cluster 1:
  Stop 0: Depot
  Stop 1: 43
  Stop 2: Depot
Total distance: 25.62 km

Optimal Route for Cluster 2:
  Stop 0: Depot
  Stop 1: 20
  Stop 2: 41
  Stop 3: 4
  Stop 4: 5
  Stop 5: 90
  Stop 6: 21
  Stop 7: 13
  Stop 8: 30
  Stop 9: 62
  Stop 10: 63
  Stop 11: 6
  Stop 12: 9
  Stop 13: 33
  Stop 14: 40
  Stop 15: 42
  Stop 16: 1
  Stop 17: 76
  Stop 18: 3
  Stop 19: 22
  Stop 20: 71
  Stop 21: 25
  Stop 22: 98
  Stop 23: 15
  Stop 24: 10
  Stop 25: 23
  Stop 26: 35
  Stop 27: 24
  Stop 28: 46
  Stop 29: 58
  Stop 30: 16
  Stop 31: 27
  Stop 32: 48
  Stop 33: 101
  Stop 34: 11
  Stop 35: 84
  Stop 36: 2
  Stop 37: 7
  Stop 38: 8
  Stop 39: 17
  Stop 40: 57
  Stop 41: Depot
Total

In [ ]:
map_output = plot_routes()

map_output.save('/content/output_map.html')

files.download('/content/output_map.html')

Total distance for Cluster 0: 42.74 km
Total distance for Cluster 1: 38.41 km
Total distance for Cluster 2: 22.04 km
Total distance for Cluster 3: 25.62 km
Total distance for Cluster 4: 37.86 km
Total distance for Cluster 5: 50.30 km
Total distance for Cluster 6: 38.59 km
Total distance for Cluster 7: 35.14 km


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>